In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# ============================================================
# main_pipeline.py - Full Integration Pipeline (Final Version)
# ICT2403 - Graphics and Image Processing
# Road Surface Damage Detection System
# ============================================================
# This is the final integrated program that runs the complete
# pipeline from start to finish in one single file.
#
# Pipeline Order:
#   Phase 1 - Frame Enhancement
#             Noise removal (Median filter)
#             Contrast enhancement (CLAHE)
#             Sharpening (Unsharp Masking for blurry frames)
#
#   Phase 2 - Segmentation (all 4 damage types)
#             Member 1 - Potholes
#               ROI-based bright/dark region detection
#               Per-frame kernel size and merge strategy
#             Member 2 - Edge Cracks
#               Left-zone HSV filtering + Adaptive threshold
#               Connected component filtering
#             Member 3 - Alligator Cracks
#               Polygon ROI + Dark region + Large closing
#             Member 4 - Raveling
#               ROI + Top-hat + Fixed threshold
#
#   Phase 3 - Final Marking
#             All 4 damage types drawn on original frame
#             Color coding: Red=Pothole, Green=Edge Crack,
#             Orange=Alligator Crack, Purple=Raveling
#
#   Phase 4 - Save Comparison Images for Report
#
# HOW TO RUN:
#   1. Make sure Frames/ folder has extracted frames
#   2. Open in Jupyter Notebook
#   3. Run all cells
#   4. All outputs saved automatically
# ============================================================


# ============================================================
# SETTINGS
# ============================================================

FRAMES_FOLDER     = "Frames/"
ENHANCED_FOLDER   = "Final_Enhanced/"
SEGMENTED_FOLDER  = "Final_Segmented/"
MARKED_FOLDER     = "Final_Marked/"
COMPARISON_FOLDER = "Final_Comparisons/"

BLUR_THRESHOLD = 400

os.makedirs(ENHANCED_FOLDER,   exist_ok=True)
os.makedirs(SEGMENTED_FOLDER,  exist_ok=True)
os.makedirs(MARKED_FOLDER,     exist_ok=True)
os.makedirs(COMPARISON_FOLDER, exist_ok=True)

print("============================================")
print("   Road Surface Damage Detection System     ")
print("   ICT2403 - Final Integration Pipeline     ")
print("============================================")
print()


# ============================================================
# Read All Frame Names
# ============================================================

all_files = os.listdir(FRAMES_FOLDER)

frame_files = []
for f in all_files:
    if f.startswith("frame_") and f.endswith(".jpg"):
        frame_files.append(f)

frame_files.sort(key=lambda x: int(x.split("_")[1].split(".")[0]))

print("Total frames found:", len(frame_files))
print()


# ============================================================
# PHASE 1: ENHANCEMENT
# ============================================================

print("--------------------------------------------")
print("PHASE 1: Frame Enhancement")
print("--------------------------------------------")

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

enhanced_count = 0
sharpened_count = 0

for frame_name in frame_files:

    img = cv2.imread(FRAMES_FOLDER + frame_name)

    if img is None:
        print("Could not read:", frame_name, "- Skipping.")
        continue

    # Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Noise removal - Median filter
    denoised = cv2.medianBlur(gray, 5)

    # Contrast enhancement - CLAHE
    contrast = clahe.apply(denoised)

    # Sharpening - only for blurry frames
    laplacian = cv2.Laplacian(contrast, cv2.CV_64F)
    sharpness = laplacian.var()

    if sharpness < BLUR_THRESHOLD:
        blurred = cv2.GaussianBlur(contrast, (5, 5), 0)
        enhanced = cv2.addWeighted(contrast, 1.5, blurred, -0.5, 0)
        sharpened_count = sharpened_count + 1
    else:
        enhanced = contrast

    cv2.imwrite(ENHANCED_FOLDER + frame_name, enhanced)
    enhanced_count = enhanced_count + 1

print("Enhancement done.")
print("Total frames enhanced:", enhanced_count)
print("Blurry frames sharpened:", sharpened_count)
print()


# ============================================================
# SEGMENTATION FUNCTIONS
# ============================================================

# ── Member 1: Pothole Segmentation ───────────────────────────

ROI_80_89   = (420, 180, 780, 440)
ROI_195_199 = (380, 180, 720, 420)
THRESHOLD_WET = 160
THRESHOLD_DRY = 80
MIN_AREA_POT  = 800

def get_pothole_settings(frame_name):
    num = int(frame_name.split("_")[1].split(".")[0])
    if num >= 80 and num <= 89:
        return ROI_80_89, THRESHOLD_WET, "BINARY", 11
    elif num == 195:
        return ROI_195_199, THRESHOLD_DRY, "BINARY_INV", 21
    elif num == 196 or num == 197:
        return ROI_195_199, THRESHOLD_DRY, "BINARY_INV", 31
    else:
        return ROI_195_199, THRESHOLD_DRY, "BINARY_INV", 41

def segment_pothole(img, frame_name):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    roi, threshold, mode, ks = get_pothole_settings(frame_name)
    x1, y1, x2, y2 = roi
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(roi_mask, (x1, y1), (x2, y2), 255, -1)
    gray_roi = cv2.bitwise_and(gray, gray, mask=roi_mask)
    if mode == "BINARY":
        _, binary = cv2.threshold(gray_roi, threshold, 255, cv2.THRESH_BINARY)
    else:
        _, binary = cv2.threshold(gray_roi, threshold, 255, cv2.THRESH_BINARY_INV)
        binary = cv2.bitwise_and(binary, binary, mask=roi_mask)
    kc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ks, ks))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kc)
    ko = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, ko)
    kd = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    final = cv2.dilate(cleaned, kd, iterations=2)
    num = int(frame_name.split("_")[1].split(".")[0])
    contours, _ = cv2.findContours(final, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    good = [c for c in contours if cv2.contourArea(c) > MIN_AREA_POT]
    if len(good) == 0:
        return np.zeros((h, w), dtype=np.uint8)
    if num >= 198 and len(good) > 1:
        all_pts = np.vstack(good)
        merged = cv2.convexHull(all_pts)
        good = [merged]
    else:
        good = [cv2.convexHull(c) for c in good]
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(mask, good, -1, 255, -1)
    return mask


# ── Member 2: Edge Crack Segmentation ────────────────────────

def segment_edge_crack(img):
    H, W = img.shape[:2]
    hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    left_width = int(W * 0.32)
    left_gray = gray[:, :left_width]
    left_hsv  = hsv[:, :left_width]
    lower_green = np.array([25, 30, 30])
    upper_green = np.array([90, 255, 255])
    grass_mask  = cv2.inRange(left_hsv, lower_green, upper_green)
    lower_sand  = np.array([0, 0, 180])
    upper_sand  = np.array([30, 50, 255])
    sand_mask   = cv2.inRange(left_hsv, lower_sand, upper_sand)
    exclude     = cv2.bitwise_or(grass_mask, sand_mask)
    blurred = cv2.GaussianBlur(left_gray, (7, 7), 0)
    binary  = cv2.adaptiveThreshold(blurred, 255,
                                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, 51, 10)
    binary  = cv2.bitwise_and(binary, cv2.bitwise_not(exclude))
    binary_full = np.zeros((H, W), dtype=np.uint8)
    binary_full[:, :left_width] = binary
    kc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
    closed = cv2.morphologyEx(binary_full, cv2.MORPH_CLOSE, kc)
    ko = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, ko)
    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(cleaned)
    for cnt in contours:
        cv2.drawContours(filled, [cnt], -1, 255, -1)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(filled)
    final_mask = np.zeros_like(filled)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= 800:
            final_mask[labels == i] = 255
    return final_mask


# ── Member 3: Alligator Crack Segmentation ───────────────────

THRESHOLD_ALIG   = 110
CLOSING_ALIG     = 31
MIN_AREA_ALIG    = 5000

def get_alig_roi(frame_name, h, w):
    num = int(frame_name.split("_")[1].split(".")[0])
    if num >= 72 and num <= 79:
        return np.array([
            (160, 160), (300, 160), (310, 230),
            (280, 340), (240, 420), (160, 478),
            (100, 478), (100, 320), (150, 220), (160, 160)
        ], np.int32)
    elif num == 90:
        return np.array([
            (160, 145), (460, 145), (470, 230),
            (450, 340), (390, 430), (180, 478),
            (120, 478), (120, 310), (150, 210), (160, 145)
        ], np.int32)
    else:
        return np.array([
            (0, 290), (330, 200), (460, 200), (470, 270),
            (450, 370), (390, 450), (200, 478), (0, 478), (0, 290)
        ], np.int32)

def segment_alligator(img, frame_name):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    roi_poly = get_alig_roi(frame_name, h, w)
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(roi_mask, [roi_poly], 255)
    gray_roi = cv2.bitwise_and(gray, gray, mask=roi_mask)
    _, binary = cv2.threshold(gray_roi, THRESHOLD_ALIG, 255, cv2.THRESH_BINARY_INV)
    binary = cv2.bitwise_and(binary, binary, mask=roi_mask)
    kc = cv2.getStructuringElement(cv2.MORPH_RECT, (CLOSING_ALIG, CLOSING_ALIG))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kc)
    ko = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, ko)
    final = cv2.bitwise_and(cleaned, cleaned, mask=roi_mask)
    return final


# ── Member 4: Raveling Segmentation ──────────────────────────

def segment_raveling(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    left_cut  = int(w * 0.12)
    right_cut = int(w * 0.80)
    top_cut   = int(h * 0.15)
    cv2.rectangle(roi_mask, (left_cut, top_cut), (right_cut, h), 255, -1)
    gray_roi = cv2.bitwise_and(gray, gray, mask=roi_mask)
    kt = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    tophat = cv2.morphologyEx(gray_roi, cv2.MORPH_TOPHAT, kt)
    _, binary = cv2.threshold(tophat, 30, 255, cv2.THRESH_BINARY)
    binary = cv2.bitwise_and(binary, binary, mask=roi_mask)
    kc = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kc)
    ko = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    final = cv2.morphologyEx(closed, cv2.MORPH_OPEN, ko)
    return final


# ============================================================
# PHASE 2: SEGMENTATION
# ============================================================

print("--------------------------------------------")
print("PHASE 2: Damage Segmentation")
print("--------------------------------------------")

segmented_count = 0

for frame_name in frame_files:

    img = cv2.imread(ENHANCED_FOLDER + frame_name)

    if img is None:
        print("Could not read enhanced frame:", frame_name, "- Skipping.")
        continue

    # Run all 4 segmentations
    pot_mask  = segment_pothole(img, frame_name)
    edge_mask = segment_edge_crack(img)
    alig_mask = segment_alligator(img, frame_name)
    rav_mask  = segment_raveling(img)

    # Save individual masks
    cv2.imwrite(SEGMENTED_FOLDER + "pothole_"   + frame_name, pot_mask)
    cv2.imwrite(SEGMENTED_FOLDER + "edgecrack_" + frame_name, edge_mask)
    cv2.imwrite(SEGMENTED_FOLDER + "alligator_" + frame_name, alig_mask)
    cv2.imwrite(SEGMENTED_FOLDER + "raveling_"  + frame_name, rav_mask)

    segmented_count = segmented_count + 1

print("Segmentation done.")
print("Total frames segmented:", segmented_count)
print()


# ============================================================
# PHASE 3: FINAL MARKING
# ============================================================

print("--------------------------------------------")
print("PHASE 3: Final Marking")
print("--------------------------------------------")

marked_count = 0

for frame_name in frame_files:

    # Load original frame for marking
    original = cv2.imread(FRAMES_FOLDER + frame_name)

    if original is None:
        continue

    # Load all masks
    pot_mask  = cv2.imread(SEGMENTED_FOLDER + "pothole_"   + frame_name, cv2.IMREAD_GRAYSCALE)
    edge_mask = cv2.imread(SEGMENTED_FOLDER + "edgecrack_" + frame_name, cv2.IMREAD_GRAYSCALE)
    alig_mask = cv2.imread(SEGMENTED_FOLDER + "alligator_" + frame_name, cv2.IMREAD_GRAYSCALE)
    rav_mask  = cv2.imread(SEGMENTED_FOLDER + "raveling_"  + frame_name, cv2.IMREAD_GRAYSCALE)

    if pot_mask is None or edge_mask is None or alig_mask is None or rav_mask is None:
        continue

    marked = original.copy()

    # Potholes - Red (0, 0, 255)
    pot_contours, _ = cv2.findContours(pot_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in pot_contours:
        if cv2.contourArea(c) > 800:
            cv2.drawContours(marked, [c], -1, (0, 0, 255), 2)

    # Edge Cracks - Green (0, 255, 0)
    edge_contours, _ = cv2.findContours(edge_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in edge_contours:
        if cv2.contourArea(c) > 800:
            cv2.drawContours(marked, [c], -1, (0, 255, 0), 2)

    # Alligator Cracks - Orange (0, 165, 255)
    alig_contours, _ = cv2.findContours(alig_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in alig_contours:
        if cv2.contourArea(c) > 5000:
            cv2.drawContours(marked, [c], -1, (0, 165, 255), 2)

    # Raveling - Purple (255, 0, 255)
    rav_contours, _ = cv2.findContours(rav_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in rav_contours:
        if cv2.contourArea(c) > 200:
            cv2.drawContours(marked, [c], -1, (255, 0, 255), 2)

    # Add color legend
    cv2.putText(marked, "Red    = Pothole",        (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255),   1)
    cv2.putText(marked, "Green  = Edge Crack",     (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0),   1)
    cv2.putText(marked, "Orange = Alligator Crack",(10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 1)
    cv2.putText(marked, "Purple = Raveling",       (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 1)

    cv2.imwrite(MARKED_FOLDER + "final_" + frame_name, marked)
    marked_count = marked_count + 1

print("Marking done.")
print("Total frames marked:", marked_count)
print()


# ============================================================
# PHASE 4: SAVE COMPARISON IMAGES FOR REPORT
# ============================================================

print("--------------------------------------------")
print("PHASE 4: Saving Report Comparison Images")
print("--------------------------------------------")

# Save 4-panel comparison for first 5 frames
comparison_frames = frame_files[:5]

for frame_name in comparison_frames:

    original  = cv2.imread(FRAMES_FOLDER    + frame_name)
    enhanced  = cv2.imread(ENHANCED_FOLDER  + frame_name, cv2.IMREAD_GRAYSCALE)
    alig_mask = cv2.imread(SEGMENTED_FOLDER + "alligator_" + frame_name, cv2.IMREAD_GRAYSCALE)
    final     = cv2.imread(MARKED_FOLDER    + "final_"     + frame_name)

    if original is None or enhanced is None or alig_mask is None or final is None:
        print("Skipping comparison for:", frame_name)
        continue

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    plt.title("Original Frame")
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(enhanced, cmap='gray')
    plt.title("Enhanced Frame")
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(alig_mask, cmap='gray')
    plt.title("Segmentation Mask\n(Alligator example)")
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(cv2.cvtColor(final, cv2.COLOR_BGR2RGB))
    plt.title("Final Marked Output\n(All damage types)")
    plt.axis('off')

    plt.suptitle("Full Pipeline Result - " + frame_name)
    plt.tight_layout()
    plt.savefig(COMPARISON_FOLDER + "pipeline_" + frame_name)
    plt.close()

print("Comparison images saved in:", COMPARISON_FOLDER)
print()


# ============================================================
# SUMMARY
# ============================================================

print("============================================")
print("   PIPELINE COMPLETE - SUMMARY              ")
print("============================================")
print()
print("Input frames processed  :", len(frame_files))
print("Enhanced frames saved   :", enhanced_count,   " ->", ENHANCED_FOLDER)
print("Segmented masks saved   :", segmented_count,  " ->", SEGMENTED_FOLDER)
print("Final marked frames     :", marked_count,     " ->", MARKED_FOLDER)
print("Comparison images saved :", len(comparison_frames), " ->", COMPARISON_FOLDER)
print()
print("Output folder structure:")
print("  Final_Enhanced/    - Enhanced frames")
print("  Final_Segmented/   - Binary masks for all 4 damage types")
print("  Final_Marked/      - Frames with all damage regions outlined")
print("  Final_Comparisons/ - 4-panel comparison images for report")
print()
print("Color coding in Final_Marked frames:")
print("  Red    = Pothole        (Member 1)")
print("  Green  = Edge Crack     (Member 2)")
print("  Orange = Alligator Crack (Member 3)")
print("  Purple = Raveling       (Member 4)")
print()
print("============================================")
print("   Road Surface Damage Detection Complete   ")
print("============================================")

   Road Surface Damage Detection System     
   ICT2403 - Final Integration Pipeline     

Total frames found: 200

--------------------------------------------
PHASE 1: Frame Enhancement
--------------------------------------------
Enhancement done.
Total frames enhanced: 200
Blurry frames sharpened: 195

--------------------------------------------
PHASE 2: Damage Segmentation
--------------------------------------------
Segmentation done.
Total frames segmented: 200

--------------------------------------------
PHASE 3: Final Marking
--------------------------------------------
Marking done.
Total frames marked: 200

--------------------------------------------
PHASE 4: Saving Report Comparison Images
--------------------------------------------
Comparison images saved in: Final_Comparisons/

   PIPELINE COMPLETE - SUMMARY              

Input frames processed  : 200
Enhanced frames saved   : 200  -> Final_Enhanced/
Segmented masks saved   : 200  -> Final_Segmented/
Final marked fra